La presente libreta recoge el código íntegro empleado para realziar el proceso de chunking de los documentos. Cabe mencionar que originalmente este proceso se dividió en varias libretas distintas, pero aquí se concatenan todas a fin de evitar múltiples archivos.

Libreta 1: extracción de entradas de Wikipedia y conversión en archivo JSON

In [1]:
!pip install wikipedia-api

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 4.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.0
    Uninstalling click-8.4.0:
      Successfully uninstalled click-8.4.0


In [2]:
# JSON Wikipedia unificado:

import wikipediaapi
import json
import os

def main():
    wiki = wikipediaapi.Wikipedia(
        user_agent="TFM-RAG-AML (tu_correo@example.com)",
        language="es",
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    entradas = {
        "Tema central": ["Lavado de dinero"],

        "Origen del dinero ilícito": [
            "Contrabando",
            "Delincuencia organizada",
            "Narcotráfico",
            "Trata de personas"
        ],

        "Economía oculta": [
            "Dinero negro",
            "Economía informal",
            "Economía sumergida",
            "Mercado negro"
        ],

        "Mecanismos de ocultación": [
            "Compañía extraterritorial",
            "Empresa fantasma",
            "Organización fachada",
            "Testaferro",
            "Pitufeo"
        ],

        "Delitos financieros": [
            "Corrupción empresarial",
            "Corrupción política",
            "Fraude",
            "Fraude bancario"
        ],

        "Fiscalidad": [
            "Elusión fiscal",
            "Evasión fiscal",
            "Fuga de capitales",
            "Paraíso fiscal"
        ],

        "Terminología AML": [
            "Conozca a su cliente",
            "Cumplimiento normativo",
            "Diligencia debida",
            "Persona expuesta políticamente",
            "Propiedad efectiva",
            "Regulación financiera"
        ],

        "Instituciones": [
            "Grupo de Acción Financiera Internacional",
            "Servicio Ejecutivo de la Comisión de Prevención del Blanqueo de Capitales e Infracciones Monetarias",
            "Unidad de Inteligencia Financiera"
        ]
    }

    secciones_excluidas = {
        "Véase también",
        "Referencias",
        "Bibliografía",
        "Enlaces externos",
        "Notas",
        "Notas y referencias"
    }

    corpus = []
    errores = []
    contador_id = 1

    for categoria, titulos in entradas.items():
        for titulo in titulos:
            documento = extraer_entrada(
                wiki=wiki,
                titulo=titulo,
                categoria=categoria,
                secciones_excluidas=secciones_excluidas,
                contador_id=contador_id
            )

            if documento:
                corpus.append(documento)
                print(f"Extraída: {titulo}")
                contador_id += 1
            else:
                errores.append(titulo)
                print(f"No encontrada o sin texto útil: {titulo}")

    os.makedirs("Datasets", exist_ok=True)

    with open("Datasets/wikipedia_aml_unificado.json", "w", encoding="utf-8") as fout:
        json.dump(corpus, fout, ensure_ascii=False, indent=2)

    with open("Datasets/wikipedia_aml_errores.json", "w", encoding="utf-8") as fout:
        json.dump(errores, fout, ensure_ascii=False, indent=2)

    print()
    print(f"Entradas extraídas: {len(corpus)}")
    print(f"Entradas con error: {len(errores)}")


def extraer_entrada(wiki, titulo, categoria, secciones_excluidas, contador_id):
    pagina = wiki.page(titulo)

    if not pagina.exists():
        return None

    textos = []

    if pagina.summary.strip():
        textos.append(pagina.summary.strip())

    for seccion in pagina.sections:
        texto_seccion = extraer_seccion(seccion, secciones_excluidas)

        if texto_seccion.strip():
            textos.append(texto_seccion.strip())

    texto_final = "\n\n".join(textos).strip()

    if not texto_final:
        return None

    return {
        "id": f"{contador_id:03}",
        "titulo": titulo,
        "categoria": categoria,
        "fuente": "Wikipedia",
        "idioma": "es",
        "tipo_documento": "entrada_enciclopedica",
        "formato_original": "html",
        "url": pagina.fullurl,
        "texto": texto_final
    }


def extraer_seccion(seccion, secciones_excluidas):
    if seccion.title in secciones_excluidas:
        return ""

    textos = []

    if seccion.text.strip():
        textos.append(seccion.title)
        textos.append(seccion.text.strip())

    for subseccion in seccion.sections:
        texto_subseccion = extraer_seccion(subseccion, secciones_excluidas)

        if texto_subseccion.strip():
            textos.append(texto_subseccion.strip())

    return "\n\n".join(textos)


main()

Extraída: Lavado de dinero
Extraída: Contrabando
Extraída: Delincuencia organizada
Extraída: Narcotráfico
Extraída: Trata de personas
Extraída: Dinero negro
Extraída: Economía informal
Extraída: Economía sumergida
Extraída: Mercado negro
Extraída: Compañía extraterritorial
Extraída: Empresa fantasma
Extraída: Organización fachada
Extraída: Testaferro
Extraída: Pitufeo
Extraída: Corrupción empresarial
Extraída: Corrupción política
Extraída: Fraude
Extraída: Fraude bancario
Extraída: Elusión fiscal
Extraída: Evasión fiscal
Extraída: Fuga de capitales
Extraída: Paraíso fiscal
Extraída: Conozca a su cliente
Extraída: Cumplimiento normativo
Extraída: Diligencia debida
Extraída: Persona expuesta políticamente
Extraída: Propiedad efectiva
Extraída: Regulación financiera
Extraída: Grupo de Acción Financiera Internacional
Extraída: Servicio Ejecutivo de la Comisión de Prevención del Blanqueo de Capitales e Infracciones Monetarias
Extraída: Unidad de Inteligencia Financiera

Entradas extraídas: 

A continuación, se incluye la posibilidad de extraer un único archivo por cada entrada.

In [3]:
# Un JSON por cada entrada


import wikipediaapi
import json
import os
import re
import unicodedata

def limpiar_nombre_archivo(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^\w\s-]", "", texto)
    texto = re.sub(r"\s+", "_", texto)
    return texto.strip("_")


def main():
    wiki = wikipediaapi.Wikipedia(
        user_agent="TFM-RAG-AML (tu_correo@example.com)",
        language="es",
        extract_format=wikipediaapi.ExtractFormat.WIKI
    )

    entradas = {
        "Tema central": ["Lavado de dinero"],

        "Origen del dinero ilícito": [
            "Contrabando",
            "Delincuencia organizada",
            "Narcotráfico",
            "Trata de personas"
        ],

        "Economía oculta": [
            "Dinero negro",
            "Economía informal",
            "Economía sumergida",
            "Mercado negro"
        ],

        "Mecanismos de ocultación": [
            "Compañía extraterritorial",
            "Empresa fantasma",
            "Organización fachada",
            "Testaferro",
            "Pitufeo"
        ],

        "Delitos financieros": [
            "Corrupción empresarial",
            "Corrupción política",
            "Fraude",
            "Fraude bancario"
        ],

        "Fiscalidad": [
            "Elusión fiscal",
            "Evasión fiscal",
            "Fuga de capitales",
            "Paraíso fiscal"
        ],

        "Terminología AML": [
            "Conozca a su cliente",
            "Cumplimiento normativo",
            "Diligencia debida",
            "Persona expuesta políticamente",
            "Propiedad efectiva",
            "Regulación financiera"
        ],

        "Instituciones": [
            "Grupo de Acción Financiera Internacional",
            "Servicio Ejecutivo de la Comisión de Prevención del Blanqueo de Capitales e Infracciones Monetarias",
            "Unidad de Inteligencia Financiera"
        ]
    }

    secciones_excluidas = {
        "Véase también",
        "Referencias",
        "Bibliografía",
        "Enlaces externos",
        "Notas",
        "Notas y referencias"
    }

    corpus = []
    errores = []
    contador_id = 1

    os.makedirs("Datasets/wikipedia_individual", exist_ok=True)

    for categoria, titulos in entradas.items():
        for titulo in titulos:
            documento = extraer_entrada(
                wiki=wiki,
                titulo=titulo,
                categoria=categoria,
                secciones_excluidas=secciones_excluidas,
                contador_id=contador_id
            )

            if documento:
                corpus.append(documento)

                nombre_limpio = limpiar_nombre_archivo(documento["titulo"])
                max_len = 60  # longitud máxima del nombre limpio
                nombre_corto = nombre_limpio[:max_len]

                ruta_salida = f"Datasets/wikipedia_individual/{documento['id']}_{nombre_corto}.json"

                with open(ruta_salida, "w", encoding="utf-8") as fout:
                    json.dump(documento, fout, ensure_ascii=False, indent=2)

                print(f"Extraída: {titulo}")
                contador_id += 1

            else:
                errores.append(titulo)
                print(f"No encontrada o sin texto útil: {titulo}")

    with open("Datasets/wikipedia_individual_errores.json", "w", encoding="utf-8") as fout:
        json.dump(errores, fout, ensure_ascii=False, indent=2)

    print()
    print(f"Entradas extraídas: {len(corpus)}")
    print(f"Entradas con error: {len(errores)}")


def extraer_entrada(wiki, titulo, categoria, secciones_excluidas, contador_id):
    pagina = wiki.page(titulo)

    if not pagina.exists():
        return None

    textos = []

    if pagina.summary.strip():
        textos.append(pagina.summary.strip())

    for seccion in pagina.sections:
        texto_seccion = extraer_seccion(seccion, secciones_excluidas)

        if texto_seccion.strip():
            textos.append(texto_seccion.strip())

    texto_final = "\n\n".join(textos).strip()

    if not texto_final:
        return None

    return {
        "id": f"{contador_id:03}",
        "titulo": titulo,
        "categoria": categoria,
        "fuente": "Wikipedia",
        "idioma": "es",
        "tipo_documento": "entrada_enciclopedica",
        "formato_original": "html",
        "url": pagina.fullurl,
        "texto": texto_final
    }


def extraer_seccion(seccion, secciones_excluidas):
    if seccion.title in secciones_excluidas:
        return ""

    textos = []

    if seccion.text.strip():
        textos.append(seccion.title)
        textos.append(seccion.text.strip())

    for subseccion in seccion.sections:
        texto_subseccion = extraer_seccion(subseccion, secciones_excluidas)

        if texto_subseccion.strip():
            textos.append(texto_subseccion.strip())

    return "\n\n".join(textos)


main()

Extraída: Lavado de dinero
Extraída: Contrabando
Extraída: Delincuencia organizada
Extraída: Narcotráfico
Extraída: Trata de personas
Extraída: Dinero negro
Extraída: Economía informal
Extraída: Economía sumergida
Extraída: Mercado negro
Extraída: Compañía extraterritorial
Extraída: Empresa fantasma
Extraída: Organización fachada
Extraída: Testaferro
Extraída: Pitufeo
Extraída: Corrupción empresarial
Extraída: Corrupción política
Extraída: Fraude
Extraída: Fraude bancario
Extraída: Elusión fiscal
Extraída: Evasión fiscal
Extraída: Fuga de capitales
Extraída: Paraíso fiscal
Extraída: Conozca a su cliente
Extraída: Cumplimiento normativo
Extraída: Diligencia debida
Extraída: Persona expuesta políticamente
Extraída: Propiedad efectiva
Extraída: Regulación financiera
Extraída: Grupo de Acción Financiera Internacional
Extraída: Servicio Ejecutivo de la Comisión de Prevención del Blanqueo de Capitales e Infracciones Monetarias
Extraída: Unidad de Inteligencia Financiera

Entradas extraídas: 

In [4]:
# Descargar JSON individualizados


import shutil
from google.colab import files

# Carpeta donde están los JSON individuales
carpeta = "Datasets/wikipedia_individual"

# Nombre del zip
zip_path = "wikipedia_individual.zip"

# Crear ZIP
shutil.make_archive("wikipedia_individual", 'zip', carpeta)

# Descargar
files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Libreta 2: extracción de recursos aplicados

In [5]:
!pip install pymupdf beautifulsoup4 lxml openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 63.9 MB/s eta 0:00:00


In [6]:
from google.colab import files

uploaded = files.upload()
excel_path = list(uploaded.keys())[0]

print("Archivo subido:", excel_path)

Saving Anexo_II_-_Tabla_Dominio_Aplicado.xlsx to Anexo_II_-_Tabla_Dominio_Aplicado.xlsx
Archivo subido: Anexo_II_-_Tabla_Dominio_Aplicado.xlsx


Aquí se sube el archivo nombrado como anexo II

In [7]:
# pipeline completo: descargar, extraer y crear JSON

import os
import re
import json
import time
import requests
import pandas as pd
import unicodedata
import fitz  # PyMuPDF

from bs4 import BeautifulSoup
from urllib.parse import urlparse


CARPETA_ORIGINALES = "dominio_aplicado_originales"
CARPETA_JSON_INDIVIDUAL = "dominio_aplicado_json_individual"
CARPETA_SALIDA = "dominio_aplicado_salida"

os.makedirs(CARPETA_ORIGINALES, exist_ok=True)
os.makedirs(CARPETA_JSON_INDIVIDUAL, exist_ok=True)
os.makedirs(CARPETA_SALIDA, exist_ok=True)


def limpiar_nombre_archivo(texto, max_len=50):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^\w\s-]", "", texto)
    texto = re.sub(r"\s+", "_", texto)
    texto = texto.strip("_")
    return texto[:max_len]


def normalizar_id(valor):
    return f"{int(valor):03}"


def normalizar_formato(valor):
    valor = str(valor).strip().lower()
    valor = valor.replace(".", "")
    return valor


def obtener_extension(formato, url):
    formato = normalizar_formato(formato)

    if formato in ["pdf", "xml", "html", "txt"]:
        return formato

    path = urlparse(url).path.lower()

    if path.endswith(".pdf"):
        return "pdf"
    elif path.endswith(".xml"):
        return "xml"
    elif path.endswith(".txt"):
        return "txt"
    else:
        return "html"


import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

def descargar_archivo(url, ruta_salida):
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; TFM-RAG-AML/1.0)"
    }

    try:
        respuesta = requests.get(
            url,
            headers=headers,
            timeout=60
        )
        respuesta.raise_for_status()

    except requests.exceptions.SSLError:
        print("Aviso: fallo SSL. Reintentando sin verificación de certificado...")

        respuesta = requests.get(
            url,
            headers=headers,
            timeout=60,
            verify=False
        )
        respuesta.raise_for_status()

    with open(ruta_salida, "wb") as f:
        f.write(respuesta.content)


def extraer_texto_pdf(ruta):
    texto_paginas = []

    with fitz.open(ruta) as doc:
        for num_pagina, pagina in enumerate(doc, start=1):
            texto = pagina.get_text("text").strip()
            if texto:
                texto_paginas.append(f"[Página {num_pagina}]\n{texto}")

    return "\n\n".join(texto_paginas).strip()


def extraer_texto_xml(ruta):
    with open(ruta, "rb") as f:
        contenido = f.read()

    soup = BeautifulSoup(contenido, "xml")
    texto = soup.get_text(separator="\n")

    lineas = [linea.strip() for linea in texto.splitlines() if linea.strip()]
    return "\n".join(lineas).strip()


def extraer_texto_html(ruta):
    with open(ruta, "rb") as f:
        contenido = f.read()

    soup = BeautifulSoup(contenido, "html.parser")

    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    texto = soup.get_text(separator="\n")
    lineas = [linea.strip() for linea in texto.splitlines() if linea.strip()]
    return "\n".join(lineas).strip()


def extraer_texto_txt(ruta):
    with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()


def extraer_texto(ruta, formato):
    formato = normalizar_formato(formato)

    if formato == "pdf":
        return extraer_texto_pdf(ruta)
    elif formato == "xml":
        return extraer_texto_xml(ruta)
    elif formato == "html":
        return extraer_texto_html(ruta)
    elif formato == "txt":
        return extraer_texto_txt(ruta)
    else:
        return ""


df = pd.read_excel(excel_path)

df.columns = [col.strip().lower() for col in df.columns]

columnas_obligatorias = [
    "id",
    "titulo",
    "categoria",
    "tipo_documento",
    "formato",
    "idioma_archivo",
    "fuente",
    "url_descarga"
]

faltantes = [col for col in columnas_obligatorias if col not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas en el Excel: {faltantes}")

corpus = []
errores = []

for _, fila in df.iterrows():
    try:
        doc_id = normalizar_id(fila["id"])
        titulo = str(fila["titulo"]).strip()
        categoria = str(fila["categoria"]).strip()
        tipo_documento = str(fila["tipo_documento"]).strip()
        formato = obtener_extension(fila["formato"], fila["url_descarga"])
        idioma = str(fila["idioma_archivo"]).strip().lower()
        fuente = str(fila["fuente"]).strip()
        url = str(fila["url_descarga"]).strip()

        nombre_corto = limpiar_nombre_archivo(titulo)
        archivo_original = f"{doc_id}_{nombre_corto}.{formato}"
        ruta_original = os.path.join(CARPETA_ORIGINALES, archivo_original)

        print(f"Descargando {doc_id}: {titulo}")

        descargar_archivo(url, ruta_original)

        texto = extraer_texto(ruta_original, formato)

        if not texto:
            raise ValueError("No se pudo extraer texto útil.")

        documento = {
            "id": doc_id,
            "titulo": titulo,
            "categoria": categoria,
            "fuente": fuente,
            "idioma": idioma,
            "tipo_documento": tipo_documento,
            "formato_original": formato,
            "url": url,
            "archivo_original": archivo_original,
            "texto": texto
        }

        corpus.append(documento)

        ruta_json_individual = os.path.join(
            CARPETA_JSON_INDIVIDUAL,
            f"{doc_id}_{nombre_corto}.json"
        )

        with open(ruta_json_individual, "w", encoding="utf-8") as f:
            json.dump(documento, f, ensure_ascii=False, indent=2)

        print(f"OK: {doc_id}")

        time.sleep(0.5)

    except Exception as e:
        error = {
            "id": str(fila.get("id", "")),
            "titulo": str(fila.get("titulo", "")),
            "url": str(fila.get("url_descarga", "")),
            "error": str(e)
        }

        errores.append(error)
        print(f"ERROR en {error['id']}: {error['titulo']}")
        print(e)


ruta_unificado = os.path.join(CARPETA_SALIDA, "dominio_aplicado_unificado.json")
ruta_errores = os.path.join(CARPETA_SALIDA, "dominio_aplicado_errores.json")

with open(ruta_unificado, "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)

with open(ruta_errores, "w", encoding="utf-8") as f:
    json.dump(errores, f, ensure_ascii=False, indent=2)

print()
print("Proceso terminado.")
print(f"Documentos procesados correctamente: {len(corpus)}")
print(f"Documentos con error: {len(errores)}")

Descargando 032: Lista consolidada sanciones UE
OK: 032
Descargando 033: Lista consolidada sanciones ONU
OK: 033
Descargando 034: ISIL (Da'esh) & Al-Qaida Sanctions List
OK: 034
Descargando 035: Comunicación informativa sobre aplicación sanciones financieras por el conflicto de Ucrania
Aviso: fallo SSL. Reintentando sin verificación de certificado...
OK: 035
Descargando 036: Sanciones financieras internacionales cuestiones generales y procedimientos especiales en materia de no proliferación
Aviso: fallo SSL. Reintentando sin verificación de certificado...
OK: 036
Descargando 037: Obligaciones de personas físicas y jurídicas en materia de sanciones financieras internacionales
Aviso: fallo SSL. Reintentando sin verificación de certificado...
OK: 037
Descargando 038: Regímenes de sanciones financieras a Irán
Aviso: fallo SSL. Reintentando sin verificación de certificado...
OK: 038
Descargando 039: Procedimiento de autorización en el ámbito financiero de sanciones a Irán
Aviso: fallo SSL. 

In [10]:
# Celda de rescate

from google.colab import files
import os
import json
import fitz
import pandas as pd
import re
import unicodedata

uploaded_manual = files.upload()

CARPETA_MANUALES = "dominio_aplicado_manuales"
os.makedirs(CARPETA_MANUALES, exist_ok=True)

for nombre, contenido in uploaded_manual.items():
    ruta = os.path.join(CARPETA_MANUALES, nombre)

    with open(ruta, "wb") as f:
        f.write(contenido)

print("Archivos manuales subidos:", list(uploaded_manual.keys()))

Saving 071_RBA_Risk_Based_Approach-Accounting-Profession.pdf to 071_RBA_Risk_Based_Approach-Accounting-Profession.pdf
Saving 072_Guidance_RBA-Trust-Company-Service-Providers.pdf to 072_Guidance_RBA-Trust-Company-Service-Providers.pdf
Saving 091_GAFILAT-Spanish-RBA-Guidance-Legal-Prof.pdf to 091_GAFILAT-Spanish-RBA-Guidance-Legal-Prof.pdf
Archivos manuales subidos: ['071_RBA_Risk_Based_Approach-Accounting-Profession.pdf', '072_Guidance_RBA-Trust-Company-Service-Providers.pdf', '091_GAFILAT-Spanish-RBA-Guidance-Legal-Prof.pdf']


In [11]:
def limpiar_nombre_archivo(texto, max_len=50):
    texto = str(texto).lower()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = re.sub(r"[^\w\s-]", "", texto)
    texto = re.sub(r"\s+", "_", texto)
    texto = texto.strip("_")
    return texto[:max_len]


def extraer_texto_pdf(ruta):
    texto_paginas = []

    with fitz.open(ruta) as doc:
        for num_pagina, pagina in enumerate(doc, start=1):
            texto = pagina.get_text("text").strip()
            if texto:
                texto_paginas.append(f"[Página {num_pagina}]\n{texto}")

    return "\n\n".join(texto_paginas).strip()


df = pd.read_excel(excel_path)
df.columns = [col.strip().lower() for col in df.columns]

corpus_manual = []
errores_manual = []

for nombre_archivo in os.listdir(CARPETA_MANUALES):
    try:
        doc_id = nombre_archivo[:3]

        fila = df[df["id"].astype(str).str.zfill(3) == doc_id]

        if fila.empty:
            raise ValueError(f"No se encontró el ID {doc_id} en el Excel.")

        fila = fila.iloc[0]

        ruta_pdf = os.path.join(CARPETA_MANUALES, nombre_archivo)
        texto = extraer_texto_pdf(ruta_pdf)

        if not texto:
            raise ValueError("No se pudo extraer texto útil del PDF.")

        titulo = str(fila["titulo"]).strip()
        nombre_corto = limpiar_nombre_archivo(titulo)

        documento = {
            "id": doc_id,
            "titulo": titulo,
            "categoria": str(fila["categoria"]).strip(),
            "fuente": str(fila["fuente"]).strip(),
            "idioma": str(fila["idioma_archivo"]).strip().lower(),
            "tipo_documento": str(fila["tipo_documento"]).strip(),
            "formato_original": "pdf",
            "url": str(fila["url_descarga"]).strip(),
            "archivo_original": nombre_archivo,
            "texto": texto
        }

        corpus_manual.append(documento)

        ruta_json = os.path.join(
            CARPETA_JSON_INDIVIDUAL,
            f"{doc_id}_{nombre_corto}.json"
        )

        with open(ruta_json, "w", encoding="utf-8") as f:
            json.dump(documento, f, ensure_ascii=False, indent=2)

        print(f"OK manual: {doc_id} - {titulo}")

    except Exception as e:
        errores_manual.append({
            "archivo": nombre_archivo,
            "error": str(e)
        })
        print(f"ERROR manual en {nombre_archivo}: {e}")


print()
print(f"Documentos manuales procesados: {len(corpus_manual)}")
print(f"Errores manuales: {len(errores_manual)}")

OK manual: 071 - Risk Based Approach Accounting Profession
OK manual: 072 - Guidance Risk Based Approach Trust Company Service Providers
OK manual: 091 - Orientación para un enfoque basado en riesgos de profesionales legales

Documentos manuales procesados: 3
Errores manuales: 0


In [12]:
# reconstruir el unificado final

import glob

corpus_final = []

for ruta_json in sorted(glob.glob(f"{CARPETA_JSON_INDIVIDUAL}/*.json")):
    with open(ruta_json, "r", encoding="utf-8") as f:
        corpus_final.append(json.load(f))

corpus_final = sorted(corpus_final, key=lambda x: int(x["id"]))

ruta_final = os.path.join(CARPETA_SALIDA, "dominio_aplicado_unificado_final.json")

with open(ruta_final, "w", encoding="utf-8") as f:
    json.dump(corpus_final, f, ensure_ascii=False, indent=2)

print(f"Corpus final reconstruido: {len(corpus_final)} documentos")
print(ruta_final)

Corpus final reconstruido: 75 documentos
dominio_aplicado_salida/dominio_aplicado_unificado_final.json


Libreta 3: fusión de ambos dominio y chunking

Primeramente se deben unir ambos dominios en un solo archivo JSON y luego se realiza la fragmentación. Los parámetros de fragmentación eran chunks de 800 tokens de longitud con un solapamiento de 120 tokens.
Todo se realiza en la siguiente celda:

In [15]:
import json
import re
import os

RUTA_WIKI = "wikipedia_aml_unificado.json"
RUTA_APLICADO = "dominio_aplicado_unificado_final.json" # Nótese que este archivo es el que en la libreta 2 se mencionaba como "unificado final".

CARPETA_SALIDA = "corpus_rag"
os.makedirs(CARPETA_SALIDA, exist_ok=True)


def cargar_json(ruta):
    with open(ruta, "r", encoding="utf-8") as f:
        return json.load(f)


def limpiar_texto(texto):
    texto = str(texto)

    texto = texto.replace("\xa0", " ")
    texto = re.sub(r"[ \t]+", " ", texto)
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    texto = re.sub(r" +\n", "\n", texto)
    texto = re.sub(r"\n +", "\n", texto)

    return texto.strip()


def chunk_por_palabras(texto, chunk_size=800, overlap=120):
    palabras = texto.split()
    chunks = []

    inicio = 0

    while inicio < len(palabras):
        fin = inicio + chunk_size
        chunk = palabras[inicio:fin]

        if chunk:
            chunks.append(" ".join(chunk))

        inicio += chunk_size - overlap

    return chunks


wiki = cargar_json(RUTA_WIKI)
aplicado = cargar_json(RUTA_APLICADO)

corpus = wiki + aplicado

corpus_normalizado = []

for doc in corpus:
    doc_normalizado = doc.copy()
    doc_normalizado["texto"] = limpiar_texto(doc.get("texto", ""))
    corpus_normalizado.append(doc_normalizado)


chunks_rag = []

for doc in corpus_normalizado:
    doc_id = doc["id"]
    texto = doc["texto"]

    chunks = chunk_por_palabras(texto, chunk_size=800, overlap=120) # Aquí los parámetros de la fragmentación

    for i, chunk in enumerate(chunks, start=1):
        chunk_obj = {
            "chunk_id": f"{doc_id}_{i:04}",
            "doc_id": doc_id,
            "titulo": doc.get("titulo", ""),
            "categoria": doc.get("categoria", ""),
            "fuente": doc.get("fuente", ""),
            "idioma": doc.get("idioma", ""),
            "tipo_documento": doc.get("tipo_documento", ""),
            "formato_original": doc.get("formato_original", ""),
            "url": doc.get("url", ""),
            "texto_chunk": chunk
        }

        chunks_rag.append(chunk_obj)


with open(f"{CARPETA_SALIDA}/corpus_aml_normalizado.json", "w", encoding="utf-8") as f:
    json.dump(corpus_normalizado, f, ensure_ascii=False, indent=2)

with open(f"{CARPETA_SALIDA}/corpus_aml_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks_rag, f, ensure_ascii=False, indent=2)


print("Documentos totales:", len(corpus_normalizado))
print("Chunks totales:", len(chunks_rag))

Documentos totales: 106
Chunks totales: 1529


Como se aprecia, de los 106 documentos totales (31 entradas de wikipedia y 75 documentos "aplicados"), se han generado 1529 chunks.

Tras la fusión, realizan pruebas para verificar que esté completo.

In [16]:
# Comprobaciones de calidad:

import json
import pandas as pd

with open("corpus_rag/corpus_aml_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

df_chunks = pd.DataFrame(chunks)

print("Total chunks:", len(df_chunks))
print("Documentos únicos:", df_chunks["doc_id"].nunique())
print()
print(df_chunks[["chunk_id", "doc_id", "titulo", "idioma", "tipo_documento", "categoria"]].head())

Total chunks: 1529
Documentos únicos: 106

   chunk_id doc_id            titulo idioma         tipo_documento  \
0  001_0001    001  Lavado de dinero     es  entrada_enciclopedica   
1  001_0002    001  Lavado de dinero     es  entrada_enciclopedica   
2  001_0003    001  Lavado de dinero     es  entrada_enciclopedica   
3  001_0004    001  Lavado de dinero     es  entrada_enciclopedica   
4  001_0005    001  Lavado de dinero     es  entrada_enciclopedica   

      categoria  
0  Tema central  
1  Tema central  
2  Tema central  
3  Tema central  
4  Tema central  


In [17]:
# longitudes

df_chunks["num_palabras"] = df_chunks["texto_chunk"].apply(lambda x: len(str(x).split()))

print(df_chunks["num_palabras"].describe())

df_chunks[["chunk_id", "doc_id", "titulo", "num_palabras", "texto_chunk"]].head(10)

count    1529.000000
mean      766.573578
std       130.273745
min         2.000000
25%       800.000000
50%       800.000000
75%       800.000000
max       800.000000
Name: num_palabras, dtype: float64


,chunk_id,doc_id,titulo,num_palabras,texto_chunk
0,001_0001,001,Lavado de dinero,800,El lavado de dinero (también conocido como lav...
1,001_0002,001,Lavado de dinero,800,tratar de darles la apariencia de recursos obt...
2,001_0003,001,Lavado de dinero,800,"en un cheque o, al menos, obtener un recibo pa..."
3,001_0004,001,Lavado de dinero,800,"instrumentos al portador, como giros postales,..."
4,001_0005,001,Lavado de dinero,800,objetivos a combatir la financiación del terro...
5,001_0006,001,Lavado de dinero,469,del narcotráfico y en la ayuda mutua. En Panam...
6,002_0001,002,Contrabando,800,"El contrabando es la entrada, la salida y la v..."
7,002_0002,002,Contrabando,800,también se aplica al paso ilegal de una fronte...
8,002_0003,002,Contrabando,615,el uso de animales entrenados.​ Una de las ven...
9,003_0001,003,Delincuencia organizada,800,La delincuencia organizada es la actividad del...


In [18]:
# los extremos más extremos:

print("Chunks más cortos:")
display(df_chunks.sort_values("num_palabras").head(10)[["chunk_id", "doc_id", "titulo", "num_palabras", "texto_chunk"]])

print("Chunks más largos:")
display(df_chunks.sort_values("num_palabras", ascending=False).head(10)[["chunk_id", "doc_id", "titulo", "num_palabras", "texto_chunk"]])

Chunks más cortos:


,chunk_id,doc_id,titulo,num_palabras,texto_chunk
802,073_0017,073,Recomendaciones sobre las medidas de control i...,2,del terrorismo.
1259,097_0008,097,Catálogo indicador de riesgo sector profesionales,4,"Prado 6, 28014, Madrid"
911,088_0004,088,Directrices modificadas del European Bank Auth...,14,clientes para tener un conocimiento adecuado d...
23,005_0005,005,Trata de personas,18,"humano. Su temática principal es, en la mayorí..."
396,034_0062,034,ISIL (Da'esh) & Al-Qaida Sanctions List,38,(QDi.204). Associated with Islamic State in Ir...
47,016_0007,016,Corrupción política,56,en la Convención de las Naciones Unidas contra...
482,057_0012,057,Orientaciones de corresponsalia bancaria,56,tercer país sujeto a requisitos en materia de ...
816,078_0002,078,Pautas para la apertura de cuentas por entidad...,75,"corresponda en cada caso, atendiendo a las esp..."
58,019_0004,019,Elusión fiscal,78,"el cumplimiento de las obligaciones fiscales, ..."
496,059_0006,059,Orientaciones joyeros,79,el comportamiento del cliente.  Riesgos relac...


Chunks más largos:


,chunk_id,doc_id,titulo,num_palabras,texto_chunk
1527,106_0097,106,Directiva UE 2024/1640 relativa a los mecanism...,800,"Artículo 70, apartado 1 Artículo 34, apartado ..."
0,001_0001,001,Lavado de dinero,800,El lavado de dinero (también conocido como lav...
1,001_0002,001,Lavado de dinero,800,tratar de darles la apariencia de recursos obt...
2,001_0003,001,Lavado de dinero,800,"en un cheque o, al menos, obtener un recibo pa..."
3,001_0004,001,Lavado de dinero,800,"instrumentos al portador, como giros postales,..."
4,001_0005,001,Lavado de dinero,800,objetivos a combatir la financiación del terro...
1526,106_0096,106,Directiva UE 2024/1640 relativa a los mecanism...,800,"apartado 4 Artículo 18 bis, apartado 2 — Artíc..."
1510,106_0080,106,Directiva UE 2024/1640 relativa a los mecanism...,800,se especificarán: a) las condiciones generales...
1509,106_0079,106,Directiva UE 2024/1640 relativa a los mecanism...,800,escrito que detalle las condiciones y los proc...
1508,106_0078,106,Directiva UE 2024/1640 relativa a los mecanism...,800,artículo 16 del Reglamento (UE) 2024/1624. Los...
